# 7.5 · 超参数调优 / Hyperparameter Tuning

> **课程定位 / Where this fits**
> 第 5 课，**Part 7 · 模型评估与优化**。
> Lesson 5, **Part 7 · Model Evaluation & Tuning**.
>
> 模型有两类参数：**参数**（权重，从数据学）和**超参数**（树深、学习率、正则强度，要人来定）。这一课讲怎么**自动搜出最好的超参数**：网格搜索、随机搜索、贝叶斯优化(Optuna)。调参是把"还行的模型"变成"很好的模型"的关键一步，也是实战和竞赛的日常。
> Models have two kinds of settings: **parameters** (weights, learned from data) and **hyperparameters** (tree depth, learning rate, regularization — set by you). This lesson covers searching for the best hyperparameters automatically: grid search, random search, Bayesian optimization (Optuna). Tuning is the key step from "okay" to "good", and a daily reality in practice and competitions.
>
> 💼 **实战/面试视角**："网格 vs 随机搜索 / 贝叶斯优化为什么更高效 / 调参怎么防泄漏" 常考。
> 💼 **Practical/interview angle:** "grid vs random search / why Bayesian is more efficient / tuning without leakage" are common.

> 💡 **面试相关 / Interview-relevant**
> - "网格搜索 vs 随机搜索的区别与取舍"（出镜率 ★★★★★）
> - "为什么随机搜索常比网格高效"（★★★★，重要超参 + 高维）
> - "贝叶斯优化的核心思想"（★★★★）
> - "调参要不要独立 test 集 / 嵌套 CV"（★★★★）

---

## 学习目标 / Learning Objectives

1. 区分参数 vs 超参数。
   Distinguish parameters vs hyperparameters.
2. 用 **GridSearchCV** 穷举搜索。
   Exhaustive search with GridSearchCV.
3. 用 **RandomizedSearchCV** 并理解它为何常更高效。
   Use RandomizedSearchCV and why it's often more efficient.
4. 用 **Optuna** 做贝叶斯优化（TPE）。
   Use Optuna for Bayesian optimization (TPE).
5. 牢记调参的**泄漏纪律**（Pipeline + 独立 test / 嵌套 CV）。
   Remember tuning's leakage discipline (Pipeline + held-out test / nested CV).

## 目录 / TOC
1. [先建直觉 + 数据 ⭐](#1)
2. [网格搜索 GridSearchCV ⭐](#2)
3. [随机搜索 + 为何更高效 ⭐](#3)
4. [贝叶斯优化 Optuna ⭐](#4)
5. [三者对比 + 防泄漏 + 小结 ⭐](#5)


<a id="1"></a>
## 1. 先建直觉 + 数据 ⭐ / Intuition & Data

调参就是在"超参数空间"里找让**验证分数最高**的那一组。三种搜索策略，区别在**怎么决定下一组试什么**：
Tuning searches the "hyperparameter space" for the combination with the highest **validation score**. Three strategies differ in **how they decide what to try next**:
- **网格搜索(Grid)**：把每个超参取几个值，**穷举所有组合**。简单、可复现，但组合数随超参个数**指数爆炸**。
  **Grid:** pick a few values per hyperparameter and **try every combination**. Simple, reproducible, but combinations **explode exponentially** with hyperparameter count.
- **随机搜索(Random)**：在空间里**随机采样**固定次数。同样预算下常常更好（下一节解释为什么）。
  **Random:** **randomly sample** a fixed number of points. Often better for the same budget (why, next section).
- **贝叶斯优化(Bayesian)**：用前面试过的结果**建一个代理模型**，智能地选下一个"最有希望"的点——少试几次就找到好解。
  **Bayesian:** builds a surrogate model from past trials to intelligently pick the next "most promising" point — finds good solutions in fewer trials.

用 **Wine**（13 特征 3 类），调一个 GBDT 的超参。
We tune a GBDT on **Wine** (13 features, 3 classes).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier
sns.set_theme(style="whitegrid")

wine = load_wine()
# 划出独立 test 集: 调参全程不碰它, 最后才用它诚实评估(7.4 的纪律) / held-out test
X_tr, X_te, y_tr, y_te = train_test_split(wine.data, wine.target, test_size=0.25,
                                          stratify=wine.target, random_state=0)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
base = GradientBoostingClassifier(random_state=0)
print(f"Wine: train {X_tr.shape}, test {X_te.shape}")
print(f"默认超参的 CV 准确率: {cross_val_score(base, X_tr, y_tr, cv=cv).mean():.4f} (调参的起点)")


<a id="2"></a>
## 2. 网格搜索 GridSearchCV ⭐ / Grid Search

`GridSearchCV` 对**所有超参组合**逐一做交叉验证，选 CV 分数最高的。它会自动用 CV（每折独立 fit），所以没有调参泄漏。缺点是组合数 = 各超参取值数的**乘积**——3 个超参各 4 个值就是 64 组，每组 5 折 = 320 次训练，超参一多就跑不动。
`GridSearchCV` cross-validates **every combination** and picks the best CV score. It uses CV internally (each fold fit independently), so no tuning leakage. The downside: combinations = the **product** of values per hyperparameter — 3 hyperparameters × 4 values = 64 combos × 5 folds = 320 fits, infeasible as hyperparameters grow.


In [ ]:
from sklearn.model_selection import GridSearchCV

# 3 个超参, 网格 = 4×3×3 = 36 组合 / a small grid
grid = {"n_estimators": [50, 100, 200, 300],
        "max_depth": [2, 3, 4],
        "learning_rate": [0.01, 0.1, 0.3]}
n_combos = np.prod([len(v) for v in grid.values()])
print(f"网格组合数: {n_combos} (= 4×3×3); 配 5 折 = {n_combos*5} 次训练")

t = time.perf_counter()
gs = GridSearchCV(base, grid, cv=cv, n_jobs=-1).fit(X_tr, y_tr)
t_grid = time.perf_counter() - t
print(f"GridSearch 用时 {t_grid:.1f}s")
print(f"最优超参: {gs.best_params_}")
print(f"最优 CV 准确率: {gs.best_score_:.4f}")


<a id="3"></a>
## 3. 随机搜索 + 为何更高效 ⭐ / Random Search & Why It Wins

**随机搜索**在超参空间里随机采样固定次数。**面试高频问题：为什么它常常比网格搜索更好？**
**Random search** samples the space randomly a fixed number of times. **A frequent question: why is it often better than grid search?**

关键洞察（Bergstra & Bengio 2012）：**通常只有少数超参真正重要**。网格搜索把预算平摊到所有超参，在不重要的超参上浪费了很多次试验；随机搜索因为每次所有超参都取不同值，等于**在重要超参上尝试了更多不同取值**。维度越高，这个优势越明显。而且随机搜索可以用**连续分布**采样（如 log-uniform 采学习率），不受网格离散点限制。
The key insight (Bergstra & Bengio 2012): **usually only a few hyperparameters truly matter**. Grid search spreads the budget over all of them, wasting trials on unimportant ones; random search varies every hyperparameter each time, effectively **trying more distinct values of the important ones**. The advantage grows with dimensionality. Random search also samples from **continuous distributions** (e.g. log-uniform for learning rate), unconstrained by grid points.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, loguniform

# 用连续/更宽的分布采样, 而非固定网格点 / sample from distributions
dist = {"n_estimators": randint(50, 350),
        "max_depth": randint(2, 6),
        "learning_rate": loguniform(1e-3, 0.5)}      # 学习率在 log 尺度均匀采样

t = time.perf_counter()
rs = RandomizedSearchCV(base, dist, n_iter=20, cv=cv, n_jobs=-1, random_state=0).fit(X_tr, y_tr)
t_rand = time.perf_counter() - t
print(f"RandomizedSearch (只试 20 次) 用时 {t_rand:.1f}s")
print(f"最优超参: {rs.best_params_}")
print(f"最优 CV 准确率: {rs.best_score_:.4f}")
print(f"\n→ 只用网格 {20/(4*3*3):.0%} 的预算, 达到相近甚至更好的结果")
print("原因: 通常只有少数超参重要, 随机搜索在重要超参上试了更多不同取值(+连续采样)")


<a id="4"></a>
## 4. 贝叶斯优化 Optuna ⭐ / Bayesian Optimization with Optuna

网格和随机搜索都是"盲搜"——每次试验**不利用前面的结果**。**贝叶斯优化**聪明在：它用试过的 (超参→分数) 建一个**代理模型**，预测"哪片区域最有希望"，然后**优先去那里试**。这样能用**更少的试验**找到好解，尤其当每次训练很贵时（大模型、大数据）价值巨大。**Optuna** 用 TPE 算法，是目前最流行的实现。
Grid and random search are "blind" — each trial **ignores previous results**. **Bayesian optimization** is smarter: it builds a **surrogate model** from past (hyperparameter→score) pairs, predicts "which region is most promising", and **focuses trials there**. This finds good solutions in **fewer trials**, hugely valuable when each training is expensive (big models/data). **Optuna** uses the TPE algorithm and is the most popular implementation.


In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)   # 关掉冗长日志 / quiet logs

# 定义目标函数: 给定一组超参, 返回 CV 分数(Optuna 来最大化它) / objective to maximize
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 350),
        "max_depth": trial.suggest_int("max_depth", 2, 5),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.5, log=True),  # log 尺度
    }
    model = GradientBoostingClassifier(random_state=0, **params)
    return cross_val_score(model, X_tr, y_tr, cv=cv).mean()

t = time.perf_counter()
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(objective, n_trials=20)                  # 同样只试 20 次 / same budget
t_opt = time.perf_counter() - t
print(f"Optuna (TPE, 20 次试验) 用时 {t_opt:.1f}s")
print(f"最优超参: {study.best_params}")
print(f"最优 CV 准确率: {study.best_value:.4f}")
print("Optuna 用历史结果引导搜索方向 → 同样预算下通常更快收敛到好解(贵模型时优势大)")


<a id="5"></a>
## 5. 三者对比 + 防泄漏 + 小结 ⭐ / Comparison, Leakage & Summary

三种方法在**同一独立 test 集**上的最终表现（这一步才用 test，全程调参没碰它）。
Final performance of all three on the **same held-out test set** (only now do we touch test — tuning never saw it).


In [ ]:
print(f"{'方法 method':<22} {'最优 CV':>9} {'test 准确率':>11}")
print(f"{'默认 default':<22} {cross_val_score(base, X_tr, y_tr, cv=cv).mean():>9.4f} {base.fit(X_tr,y_tr).score(X_te,y_te):>11.4f}")
print(f"{'GridSearch(36组)':<22} {gs.best_score_:>9.4f} {gs.score(X_te, y_te):>11.4f}")
print(f"{'RandomSearch(20次)':<22} {rs.best_score_:>9.4f} {rs.score(X_te, y_te):>11.4f}")
best_opt = GradientBoostingClassifier(random_state=0, **study.best_params).fit(X_tr, y_tr)
print(f"{'Optuna(20次)':<22} {study.best_value:>9.4f} {best_opt.score(X_te, y_te):>11.4f}")
print("\n💡 防泄漏纪律:")
print("  1. 超参在 CV 内选(GridSearchCV/RandomizedSearchCV 自动); 预处理放进 Pipeline(3.12)")
print("  2. 用一个'全程不参与调参'的 test 集报告最终性能(本课做法)")
print("  3. 要无偏地比较'调参方法本身', 用嵌套 CV(7.4)")


```
参数(权重, 数据学) vs 超参数(树深/学习率/正则, 人来定+搜索)
网格搜索: 穷举所有组合; 简单可复现, 但组合数随超参指数爆炸
随机搜索: 随机采样固定次数; 同预算常更优(少数超参重要+可连续采样); 高维优势大
贝叶斯优化(Optuna/TPE): 用历史结果建代理模型引导搜索, 少试几次找到好解(贵模型首选)
防泄漏: 超参在 CV 内选(Pipeline 防预处理泄漏); 留独立 test 报性能; 嵌套 CV 比较调参方法
```

### 💡 面试速查 / Interview cheat-sheet
1. **网格穷举(指数爆炸) vs 随机采样(同预算常更优)**。
   Grid is exhaustive (exponential blow-up); random often wins for the same budget.
2. **随机为何更高效**: 少数超参重要, 随机在它们上试了更多取值 + 连续采样。
   Random wins because few hyperparameters matter; it tries more values of them + continuous sampling.
3. **贝叶斯优化用历史引导搜索**, 试验昂贵时最值(Optuna/TPE)。
   Bayesian optimization uses history to guide search; best when trials are expensive.
4. **调参在 CV 内 + 留独立 test**; 比较调参方法用嵌套 CV。
   Tune inside CV + keep a held-out test; compare tuning methods with nested CV.
5. **学习率等用 log 尺度搜索**(log-uniform)。
   Search learning-rate-like params on a log scale (log-uniform).

### 下一节 / Next
**7.6 多目标与帕累托**——现实里常要同时优化多个相互冲突的目标(精度 vs 速度, 精度 vs 公平)。帕累托前沿告诉你"哪些权衡是最优的"。
**7.6 Multi-objective & Pareto** — real problems optimize several conflicting objectives at once (accuracy vs speed, accuracy vs fairness). The Pareto frontier shows which trade-offs are optimal.
